# Where is the Google Merchandise Store losing revenue?

This notebook answers the project's business question from the dbt marts: where conversion and revenue fall short across devices, channels, visitor types, the checkout funnel, products and customers, whether those gaps are statistically real, and what closing part of them would be worth.

**Data:** Google Analytics 4 sample e-commerce export for the Google Merchandise Store, 1 November 2020 to 31 January 2021 (92 days), modelled in BigQuery with dbt and exported by `analysis/export_marts.py` to `data/exports/`.

**Definitions** (full list in `docs/business_case.md`):
- **Conversion rate:** sessions with a purchase event / sessions.
- **Order value:** revenue of a de-duplicated order. Purchases without a usable transaction ID are excluded from orders and revenue (see the reconciliation below).
- **Revenue per conversion:** order revenue / sessions with a purchase. It is slightly below order value because unidentified purchases count as conversions but carry no revenue, which keeps the dollar estimates conservative.
- **Funnel steps** use a closed funnel: a session counts at a step only if it also completed every earlier step. GA4 does not force steps into order, so an open count can show more sessions at a later step than an earlier one.
- **Cart abandonment:** sessions that added to cart without purchasing in the same session / sessions that added to cart.

**Statistics:** 95% Wilson intervals for rates, two-proportion z-tests for differences with Holm adjustment within each family of tests, and bootstrap intervals with Mann-Whitney tests for order values. Helpers are in `analysis/conversion_stats.py`.

Run from the repository root: `jupyter nbconvert --to notebook --execute --inplace notebooks/conversion_analysis.ipynb`

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "analysis"))
import conversion_stats as cs

DATA = Path(os.environ.get("EXPORT_DIR", ROOT / "data" / "exports"))
OUT = DATA / "analysis"
IMAGES = Path(os.environ.get("IMAGE_DIR", ROOT / "images"))
OUT.mkdir(parents=True, exist_ok=True)
IMAGES.mkdir(parents=True, exist_ok=True)

sessions = pd.read_csv(DATA / "daily_sessions.csv", parse_dates=["session_date"])
sessions["month"] = sessions["session_date"].dt.to_period("M").astype(str)
orders = pd.read_csv(DATA / "orders.csv", parse_dates=["order_date"])
products = pd.read_csv(DATA / "product_monthly.csv")
buyers = pd.read_csv(DATA / "buyers.csv")
segments = pd.read_csv(DATA / "customer_segments.csv")
recon = pd.read_csv(DATA / "purchase_reconciliation.csv")

pd.options.display.float_format = "{:,.4f}".format
pct = lambda x: f"{x:.2%}"
pp = lambda x: f"{x * 100:+.2f} pp"
usd = lambda x: f"${x:,.0f}"

# Chart style: one recessive grid, text in neutral ink, three validated series colours.
INK, INK_2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e6e5e0", "#fcfcfb"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "axes.titlecolor": INK, "axes.titlesize": 12,
    "axes.titleweight": "semibold", "axes.titlelocation": "left", "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": INK_2, "ytick.color": INK_2, "text.color": INK, "font.size": 10, "legend.frameon": False,
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})
print(f"{sessions['sessions'].sum():,} sessions, {len(orders):,} orders, "
      f"{sessions['session_date'].min():%d %b %Y} to {sessions['session_date'].max():%d %b %Y}")

## 1. Purchase data quality

Before measuring revenue, raw purchase events are split into counted orders, repeat events for an order already counted, and purchases with no usable transaction ID. Only counted orders enter revenue, order value and the dollar estimates.

In [ ]:
r = recon.sum(numeric_only=True)
quality = pd.DataFrame({
    "events": [r.purchase_events, r.orders, r.duplicate_events, r.unidentified_events],
    "revenue_usd": [r.raw_purchase_revenue_usd, r.order_revenue_usd, r.duplicate_revenue_usd, r.unidentified_revenue_usd],
}, index=["All purchase events", "Counted orders", "Duplicate events", "No usable transaction ID"])
quality["share_of_raw_revenue"] = quality["revenue_usd"] / r.raw_purchase_revenue_usd
in_sessions = sessions["revenue_usd"].sum()
print(f"Counted order revenue: {usd(r.order_revenue_usd)}; of which placed in a tracked session: {usd(in_sessions)} "
      f"({in_sessions / r.order_revenue_usd:.1%}).")
quality

## 2. Headline numbers

In [ ]:
def kpis(t):
    t = t.copy()
    t["conversion_rate"] = t["purchase_sessions"] / t["sessions"]
    t["aov"] = t["revenue_usd"] / t["orders"]
    t["revenue_per_session"] = t["revenue_usd"] / t["sessions"]
    return t

cols = ["sessions", "purchase_sessions", "orders", "revenue_usd", "add_to_cart_sessions", "cart_purchase_sessions"]
overall = kpis(sessions[cols].sum().to_frame().T)
monthly = cs.rate_table(sessions, "month").sort_values("month")
SITE_RATE = overall["conversion_rate"].iloc[0]
SITE_AOV = overall["aov"].iloc[0]
display(overall.style.format({"sessions": "{:,.0f}", "purchase_sessions": "{:,.0f}", "orders": "{:,.0f}",
                              "add_to_cart_sessions": "{:,.0f}", "revenue_usd": "${:,.0f}",
                              "conversion_rate": "{:.2%}", "aov": "${:,.2f}", "revenue_per_session": "${:,.2f}"}))
monthly[["month", "sessions", "purchase_sessions", "rate", "ci_low", "ci_high", "orders", "revenue_usd", "aov"]]

## 3. Conversion by segment

Each dot is a segment's conversion rate with its 95% interval. The dashed line is the site average. Channel is the session's own source where GA4 recorded one, otherwise the visitor's first-touch source; *Unknown* is traffic whose source the public sample obfuscates.

In [ ]:
DIMENSIONS = {"device_category": "Device", "channel": "Channel", "user_type": "New or returning", "country_group": "Country"}
seg_tables = []
for dim, label in DIMENSIONS.items():
    t = cs.rate_table(sessions, dim).rename(columns={dim: "segment"})
    t.insert(0, "dimension", label)
    seg_tables.append(t)
segment_rates = pd.concat(seg_tables, ignore_index=True)
segment_rates.to_csv(OUT / "segment_conversion.csv", index=False)

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, (label, t) in zip(axes.flat, segment_rates.groupby("dimension", sort=False)):
    t = t.sort_values("rate")
    y = np.arange(len(t))
    ax.errorbar(t["rate"], y, xerr=[t["rate"] - t["ci_low"], t["ci_high"] - t["rate"]], fmt="o",
                color=SERIES[0], ecolor=SERIES[0], elinewidth=2, capsize=0, markersize=8,
                markeredgecolor=SURFACE, markeredgewidth=2)
    for yi, (_, row) in zip(y, t.iterrows()):
        ax.annotate(f"{row['rate']:.2%}  ({row['sessions']:,.0f} sessions)", (row["ci_high"], yi),
                    xytext=(6, 0), textcoords="offset points", va="center", fontsize=8.5, color=INK_2,
                    bbox=dict(boxstyle="square,pad=0.15", fc=SURFACE, ec="none"))
    ax.axvline(SITE_RATE, color=INK_2, linestyle="--", linewidth=1)
    ax.set_yticks(y, t["segment"])
    ax.set_title(label)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
    ax.set_xlim(0, t["ci_high"].max() * 1.6)
    ax.grid(axis="y", visible=False)
fig.suptitle("Session conversion rate with 95% confidence intervals", x=0.01, ha="left", fontsize=13, fontweight="semibold")
fig.tight_layout()
fig.savefig(IMAGES / "analysis_conversion_by_segment.png")
plt.show()
segment_rates[["dimension", "segment", "sessions", "purchase_sessions", "rate", "ci_low", "ci_high", "aov", "revenue_per_session"]]

## 4. Which differences are real?

Devices are compared with desktop and returning visitors with new ones. Channels and countries have no natural baseline, so each is compared with all other sessions. p-values are Holm-adjusted within each dimension.

In [ ]:
tests = cs.holm(pd.concat([
    cs.compare_segments(sessions, "device_category", reference="desktop"),
    cs.compare_segments(sessions, "user_type", reference="New"),
    cs.compare_segments(sessions, "channel"),
    cs.compare_segments(sessions, "country_group"),
    cs.compare_segments(sessions, "month", reference="2020-11"),
], ignore_index=True))
tests.to_csv(OUT / "significance_tests.csv", index=False)
tests[["dimension", "segment", "compared_with", "segment_rate", "comparison_rate", "difference",
       "diff_ci_low", "diff_ci_high", "relative_difference", "p_adjusted", "significant"]]

## 5. Where the funnel leaks

Closed funnel: each rate is the share of sessions at one step that went on to the next. In this sample the shipping step fires together with the start of checkout, so that step shows 100%.

Device steps are tested against desktop and new visitors' steps against returning visitors'. The chart shows the comparison with the larger gaps.

In [ ]:
funnel_all = cs.funnel_table(sessions)
funnel_device = cs.funnel_table(sessions, "device_category")
funnel_user = cs.funnel_table(sessions, "user_type")
step_tests = cs.funnel_step_tests(sessions, "device_category", reference="desktop")
user_step_tests = cs.funnel_step_tests(sessions, "user_type", reference="Returning")
funnel_all.to_csv(OUT / "funnel_overall.csv", index=False)
pd.concat([funnel_device, funnel_user]).to_csv(OUT / "funnel_by_segment.csv", index=False)
pd.concat([step_tests, user_step_tests]).to_csv(OUT / "funnel_step_tests.csv", index=False)

steps = funnel_all[funnel_all["step_number"] > 0]
biggest_leak = steps.loc[steps["lost_from_previous"].idxmax()]
lowest_step = steps.loc[steps["rate_from_previous"].idxmin()]
purchases_skipping = 1 - funnel_all["sessions_reaching"].iloc[-1] / sessions["purchase_sessions"].sum()

device_gap = step_tests["difference"].abs().max()
user_gap = user_step_tests["difference"].abs().max()
if user_gap >= device_gap:
    chart_data, chart_segments, chart_label = funnel_user, ["Returning", "New"], "returning vs new visitors"
else:
    chart_data, chart_segments, chart_label = funnel_device, ["desktop", "mobile"], "desktop vs mobile"
plot = chart_data[(chart_data["step_number"] > 0) & chart_data["segment"].isin(chart_segments)]
labels = list(plot["step"].unique())
y = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(9, 4.8))
for i, seg in enumerate(chart_segments):
    vals = plot[plot["segment"] == seg].set_index("step").loc[labels, "rate_from_previous"]
    bars = ax.barh(y + (i - 0.5) * 0.38, vals, height=0.36, color=SERIES[i], label=seg.title(),
                   edgecolor=SURFACE, linewidth=2)
    ax.bar_label(bars, labels=[f"{v:.0%}" for v in vals], padding=4, fontsize=8.5, color=INK_2)
ax.set_yticks(y, [f"{a_}  →  {b_}" for a_, b_ in zip(["Session"] + labels[:-1], labels)])
ax.invert_yaxis()
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlim(0, 1.12)
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right")
ax.set_title(f"Share of sessions continuing to the next step, {chart_label}")
fig.savefig(IMAGES / "analysis_funnel.png")
plt.show()
print(f"{purchases_skipping:.0%} of sessions with a purchase skipped an earlier step in the same session "
      "(for example, the product was viewed in an earlier visit), so the closed funnel ends below total purchases.")
display(funnel_all)
cols_ = ["segment", "step", "segment_rate", "comparison_rate", "difference", "diff_ci_low", "diff_ci_high", "p_adjusted", "significant"]
display(user_step_tests[cols_])
step_tests[cols_]

## 6. Cart abandonment

Share of sessions that added to cart but did not purchase.

In [ ]:
abandon = []
for dim, label in DIMENSIONS.items():
    t = cs.rate_table(sessions, dim, num="cart_purchase_sessions", den="add_to_cart_sessions").rename(columns={dim: "segment"})
    t.insert(0, "dimension", label)
    t["abandonment_rate"] = 1 - t["rate"]
    t["abandon_ci_low"], t["abandon_ci_high"] = 1 - t["ci_high"], 1 - t["ci_low"]
    abandon.append(t)
abandon = pd.concat(abandon, ignore_index=True)
cart_tests = cs.holm(pd.concat([
    cs.compare_segments(sessions, "device_category", num="cart_purchase_sessions", den="add_to_cart_sessions", reference="desktop", family="cart"),
    cs.compare_segments(sessions, "user_type", num="cart_purchase_sessions", den="add_to_cart_sessions", reference="New", family="cart"),
], ignore_index=True))
abandon.to_csv(OUT / "cart_abandonment.csv", index=False)
SITE_ABANDON = 1 - overall["cart_purchase_sessions"].iloc[0] / overall["add_to_cart_sessions"].iloc[0]
print(f"Site cart abandonment: {SITE_ABANDON:.1%}")
display(abandon[["dimension", "segment", "add_to_cart_sessions", "abandonment_rate", "abandon_ci_low", "abandon_ci_high"]])
cart_tests[["segment", "compared_with", "segment_rate", "comparison_rate", "difference", "p_adjusted", "significant"]]

## 7. Order value

Order values are skewed by a few large orders, so means get bootstrap intervals and differences are tested with Mann-Whitney, which compares the whole distribution rather than only the mean. Orders whose session was not tracked are left out of the device and visitor comparisons.

In [ ]:
tracked = orders[orders["device_category"] != "No session"]
aov_tests = pd.concat([
    cs.compare_order_values(tracked, "device_category", reference="desktop"),
    cs.compare_order_values(tracked, "user_type", reference="New"),
    cs.compare_order_values(tracked, "channel", reference="Direct"),
], ignore_index=True)
aov_tests.to_csv(OUT / "order_value_tests.csv", index=False)

dev_aov = aov_tests[aov_tests["dimension"] == "device_category"].sort_values("mean_order_value")
fig, ax = plt.subplots(figsize=(7.5, 2.8))
yy = np.arange(len(dev_aov))
ax.errorbar(dev_aov["mean_order_value"], yy,
            xerr=[dev_aov["mean_order_value"] - dev_aov["mean_ci_low"], dev_aov["mean_ci_high"] - dev_aov["mean_order_value"]],
            fmt="o", color=SERIES[0], elinewidth=2, capsize=0, markersize=8, markeredgecolor=SURFACE, markeredgewidth=2)
for yi, (_, row) in zip(yy, dev_aov.iterrows()):
    ax.annotate(f"${row['mean_order_value']:,.0f}  ({row['orders']:,} orders)", (row["mean_ci_high"], yi),
                xytext=(6, 0), textcoords="offset points", va="center", fontsize=8.5, color=INK_2,
                    bbox=dict(boxstyle="square,pad=0.15", fc=SURFACE, ec="none"))
ax.set_yticks(yy, dev_aov["segment"].str.title())
ax.set_xlim(0, dev_aov["mean_ci_high"].max() * 1.4)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.grid(axis="y", visible=False)
ax.set_title("Mean order value by device, with 95% bootstrap intervals")
fig.savefig(IMAGES / "analysis_order_value_by_device.png")
plt.show()
aov_tests[["dimension", "segment", "orders", "mean_order_value", "mean_ci_low", "mean_ci_high", "median_order_value",
           "compared_with", "difference", "diff_ci_low", "diff_ci_high", "p_adjusted", "significant"]]

## 8. Products

This section uses purchases only. In this sample, `view_item` and `add_to_cart` events carry long lists of products, so per-product view and add counts measure appearances in lists rather than a visitor looking at or adding that product. The cell below shows the size of the lists. Purchases are reliable: item revenue matches order revenue.

Categories are the merchandise groups assigned in dbt (`docs/data_model.md`), since the raw category labels differ between event types.

In [ ]:
items_per_view_session = products["item_views"].sum() / sessions["view_item_sessions"].sum()
items_per_cart_session = products["add_to_carts"].sum() / sessions["add_to_cart_sessions"].sum()
print(f"Item-list rows per session: {items_per_view_session:.0f} on product views, {items_per_cart_session:.0f} on adds to cart. "
      f"Item revenue {usd(products['item_revenue_usd'].sum())} vs order revenue {usd(orders['revenue_usd'].sum())}.")

prod = products.groupby(["item_name", "item_category"], as_index=False)[["orders_with_item", "units_sold", "item_revenue_usd"]].sum()
prod = prod[prod["item_revenue_usd"] > 0].sort_values("item_revenue_usd", ascending=False).reset_index(drop=True)
prod["share_of_revenue"] = prod["item_revenue_usd"] / prod["item_revenue_usd"].sum()
prod["cumulative_share"] = prod["share_of_revenue"].cumsum()
prod["avg_unit_price"] = prod["item_revenue_usd"] / prod["units_sold"]

cat = prod.groupby("item_category", as_index=False).agg(
    products=("item_name", "nunique"), orders_with_item=("orders_with_item", "sum"),
    units_sold=("units_sold", "sum"), item_revenue_usd=("item_revenue_usd", "sum"))
cat["share_of_revenue"] = cat["item_revenue_usd"] / cat["item_revenue_usd"].sum()
cat["share_of_orders"] = cat["orders_with_item"] / len(orders)
cat["avg_unit_price"] = cat["item_revenue_usd"] / cat["units_sold"]
cat = cat.sort_values("item_revenue_usd", ascending=False).reset_index(drop=True)

monthly_cat = products.groupby(["month", "item_category"], as_index=False)["item_revenue_usd"].sum()
monthly_cat = monthly_cat.pivot(index="item_category", columns="month", values="item_revenue_usd").fillna(0)
monthly_cat = monthly_cat.loc[cat["item_category"]]

PRODUCTS_FOR_HALF = int((prod["cumulative_share"] < 0.5).sum() + 1)
TOP20_SHARE = prod.head(20)["share_of_revenue"].sum()
items_per_order = orders["item_quantity"].mean()
cat.to_csv(OUT / "category_performance.csv", index=False)
prod.to_csv(OUT / "product_revenue.csv", index=False)
print(f"{len(prod)} products sold. The top {PRODUCTS_FOR_HALF} brought half of item revenue; the top 20 brought {TOP20_SHARE:.0%}. "
      f"Orders averaged {items_per_order:.1f} units.")

top = cat.head(8).iloc[::-1]
fig, ax = plt.subplots(figsize=(8.5, 3.6))
yy = np.arange(len(top))
bars = ax.barh(yy, top["item_revenue_usd"], height=0.55, color=SERIES[0], edgecolor=SURFACE, linewidth=2)
ax.bar_label(bars, labels=[f"${v:,.0f}  ({s_:.0%})" for v, s_ in zip(top["item_revenue_usd"], top["share_of_revenue"])],
             padding=4, fontsize=8.5, color=INK_2)
ax.set_yticks(yy, top["item_category"])
ax.set_xlim(0, top["item_revenue_usd"].max() * 1.35)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v / 1000:,.0f}k"))
ax.grid(axis="y", visible=False)
ax.set_title("Item revenue by merchandise group, November 2020 to January 2021")
fig.savefig(IMAGES / "analysis_category_revenue.png")
plt.show()
display(cat)
display(monthly_cat.style.format("${:,.0f}"))
prod.head(15)[["item_name", "item_category", "orders_with_item", "units_sold", "item_revenue_usd", "share_of_revenue", "avg_unit_price"]]

## 9. Customers

Three months is too short for lifetime value or cohort retention, so this section describes how concentrated revenue is among repeat buyers and which first-touch channels bring visitors who go on to buy.

Channels are also judged two ways. Per session, each visit is credited to the channel it came from. By first touch, a visitor is credited to the channel that first brought them, and the question is whether they ever bought in the three months. A channel that mostly brings people who buy on a later visit looks weak per session and strong by first touch.

In [ ]:
repeat_rate = buyers["is_repeat_buyer"].mean()
repeat_rev_share = buyers.loc[buyers["is_repeat_buyer"], "revenue_usd"].sum() / buyers["revenue_usd"].sum()
top10 = buyers.sort_values("revenue_usd", ascending=False)
top10_share = top10.head(int(np.ceil(len(top10) * 0.10)))["revenue_usd"].sum() / top10["revenue_usd"].sum()
print(f"{len(buyers):,} buyers; {repeat_rate:.1%} ordered more than once and they brought {repeat_rev_share:.1%} of revenue. "
      f"The top 10% of buyers brought {top10_share:.1%}.")

seg_summary = segments.groupby("customer_segment", as_index=False)[["users", "buyers", "orders", "revenue_usd"]].sum()
seg_summary["share_of_revenue"] = seg_summary["revenue_usd"] / seg_summary["revenue_usd"].sum()
channel_buyers = segments.groupby("first_user_channel", as_index=False)[["users", "buyers", "revenue_usd"]].sum()
channel_buyers["visitor_to_buyer"] = channel_buyers["buyers"] / channel_buyers["users"]
channel_buyers["ci_low"], channel_buyers["ci_high"] = cs.wilson(channel_buyers["buyers"], channel_buyers["users"])
channel_buyers["revenue_per_visitor"] = channel_buyers["revenue_usd"] / channel_buyers["users"]
channel_buyers = channel_buyers.sort_values("users", ascending=False)
channel_buyers.to_csv(OUT / "first_channel_buyers.csv", index=False)

# Same channels judged two ways: per session, and by whether visitors they first brought ever bought.
session_view = cs.rate_table(sessions, "channel")[["channel", "sessions", "rate"]].rename(
    columns={"channel": "channel", "rate": "session_conversion"})
attribution = session_view.merge(channel_buyers[["first_user_channel", "users", "visitor_to_buyer", "ci_low", "ci_high"]]
                                 .rename(columns={"first_user_channel": "channel", "users": "first_touch_visitors"}),
                                 on="channel", how="inner")
attribution.to_csv(OUT / "channel_attribution.csv", index=False)
print("Channels judged per session versus by the visitors they first brought:")
display(attribution)
display(seg_summary)
channel_buyers

## 10. What closing the gaps would be worth

Each estimate takes a gap found significant above and asks what the store would gain over these three months if part of it closed. Assumptions:
- **Scenarios:** 10%, 25% (central) and 50% of the gap closed. Closing the whole gap is not assumed: returning visitors arrive with more intent, so part of their advantage is who they are, not what the site does.
- **Value of a conversion:** each segment's own revenue per converting session.
- **Funnel steps:** extra sessions passing a step go on to purchase at that segment's own rate from that step onward.
- **New visitors' carts:** the share of new visitors' carts that end in a purchase in the same session, moved toward returning visitors' share.
- **Not sized:** paid search (it converts poorly per session but its visitors buy as often as organic search's later on; see section 9), and products (view and add data are not reliable per product; see section 8). The dataset has no costs, so these are revenue estimates, not profit.
- Estimates overlap (a better checkout is part of better cart completion), so they are alternatives, not a total. November to January includes the holiday peak, so annualising would overstate them.

In [ ]:
FRACTIONS = (0.10, 0.25, 0.50)
opps = []

def step_opportunities(tests_, funnel_, by, reference):
    rates = cs.rate_table(sessions, by).set_index(by)
    fx = funnel_.set_index(["segment", "step"])
    for _, t in tests_[(tests_["significant"]) & (tests_["difference"] < 0)].iterrows():
        seg = t["segment"]
        prev_step, step = t["step"].split(" -> ")
        trials = fx.loc[(seg, prev_step), "sessions_reaching"]
        reaching = fx.loc[(seg, step), "sessions_reaching"]
        last = fx.loc[(seg, cs.FUNNEL[-1][1]), "sessions_reaching"]
        onward = last / reaching if reaching else 0
        value = rates.loc[seg, "revenue_per_conversion"] * min(onward, 1)
        opps.append({"opportunity": f"{seg.title()} visitors: {prev_step.lower()} to {step.lower()}",
                     "lever": "Funnel step", "segment": seg, "baseline_rate": t["segment_rate"],
                     "target_rate": t["comparison_rate"], "trials": trials,
                     **cs.size_gap(trials, t["segment_rate"], t["comparison_rate"], value, FRACTIONS)})

dev = cs.rate_table(sessions, "device_category").set_index("device_category")
dev_tests = tests[tests["dimension"] == "device_category"].set_index("segment")
for seg, row in dev.iterrows():
    if seg != "desktop" and seg in dev_tests.index and dev_tests.loc[seg, "significant"] and dev_tests.loc[seg, "difference"] < 0:
        opps.append({"opportunity": f"{seg.title()} conversion closer to desktop", "lever": "Whole session",
                     "segment": seg, "baseline_rate": row["rate"], "target_rate": dev.loc["desktop", "rate"],
                     "trials": row["sessions"],
                     **cs.size_gap(row["sessions"], row["rate"], dev.loc["desktop", "rate"], row["revenue_per_conversion"], FRACTIONS)})
step_opportunities(step_tests, funnel_device, "device_category", "desktop")
step_opportunities(user_step_tests, funnel_user, "user_type", "Returning")

ut = cs.rate_table(sessions, "user_type", num="cart_purchase_sessions", den="add_to_cart_sessions").set_index("user_type")
ct = cart_tests[cart_tests["dimension"] == "user_type"]
if {"New", "Returning"} <= set(ut.index) and len(ct) and ct.iloc[0]["significant"] and ut.loc["New", "rate"] < ut.loc["Returning", "rate"]:
    new = ut.loc["New"]
    opps.append({"opportunity": "New visitors' carts completed closer to returning visitors' rate", "lever": "Cart",
                 "segment": "New", "baseline_rate": new["rate"], "target_rate": ut.loc["Returning", "rate"],
                 "trials": new["add_to_cart_sessions"],
                 **cs.size_gap(new["add_to_cart_sessions"], new["rate"], ut.loc["Returning", "rate"],
                               new["revenue_per_conversion"], FRACTIONS)})

opportunities = pd.DataFrame(opps).sort_values("extra_revenue_25pct", ascending=False).reset_index(drop=True)
opportunities.to_csv(OUT / "opportunities.csv", index=False)
period_revenue = r.order_revenue_usd

if len(opportunities):
    o = opportunities.head(8).iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, 0.55 * len(o) + 1.2))
    yy = np.arange(len(o))
    ax.barh(yy, o["extra_revenue_25pct"], height=0.5, color=SERIES[0], edgecolor=SURFACE, linewidth=2)
    ax.errorbar(o["extra_revenue_25pct"], yy, xerr=[o["extra_revenue_25pct"] - o["extra_revenue_10pct"],
                o["extra_revenue_50pct"] - o["extra_revenue_25pct"]], fmt="none", ecolor=INK_2, elinewidth=1.5)
    for yi, (_, row) in zip(yy, o.iterrows()):
        ax.annotate(f"${row['extra_revenue_25pct']:,.0f}  (${row['extra_revenue_10pct']:,.0f} to ${row['extra_revenue_50pct']:,.0f})",
                    (row["extra_revenue_50pct"], yi), xytext=(6, 0), textcoords="offset points", va="center", fontsize=8.5, color=INK_2,
                    bbox=dict(boxstyle="square,pad=0.15", fc=SURFACE, ec="none"))
    ax.set_yticks(yy, o["opportunity"])
    ax.set_xlim(0, o["extra_revenue_50pct"].max() * 1.6)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
    ax.grid(axis="y", visible=False)
    ax.set_title("Extra revenue over the three months if 25% of each gap closed (whiskers: 10% to 50%)")
    fig.savefig(IMAGES / "analysis_opportunities.png")
    plt.show()
print(f"For scale: counted order revenue in the period was {usd(period_revenue)}.")
opportunities

## 11. Summary

Generated from the results above and saved to `data/exports/analysis/summary.md`.

In [ ]:
def test_line(dim, seg):
    t = tests[(tests["dimension"] == dim) & (tests["segment"] == seg)]
    if t.empty:
        return "not tested"
    t = t.iloc[0]
    return (f"{pct(t['segment_rate'])} vs {pct(t['comparison_rate'])} ({t['compared_with']}), difference {pp(t['difference'])} "
            f"[95% CI {pp(t['diff_ci_low'])} to {pp(t['diff_ci_high'])}], adjusted p = {t['p_adjusted']:.3g}"
            f"{'' if t['significant'] else ', not significant'}")

ch = tests[tests["dimension"] == "channel"].sort_values("segment_rate")
sig_ch = ch[ch["significant"]]
lines = [
    f"- **Overall:** {overall['sessions'].iloc[0]:,.0f} sessions, conversion {pct(SITE_RATE)}, {overall['orders'].iloc[0]:,.0f} orders, "
    f"{usd(overall['revenue_usd'].iloc[0])} revenue in tracked sessions, mean order {usd(SITE_AOV)}, revenue per session ${overall['revenue_per_session'].iloc[0]:.2f}.",
    f"- **Data quality:** {quality.loc['Duplicate events', 'share_of_raw_revenue']:.1%} of raw purchase revenue was duplicate events and "
    f"{quality.loc['No usable transaction ID', 'share_of_raw_revenue']:.1%} had no usable transaction ID; both are excluded from revenue.",
    f"- **By month:** " + "; ".join(f"{m.month} {pct(m.rate)}" for m in monthly.itertuples()) + ".",
]
for seg in [s for s in dev.index if s != "desktop"]:
    lines.append(f"- **{seg.title()} vs desktop:** {test_line('device_category', seg)}.")
lines.append(f"- **Returning vs new:** {test_line('user_type', 'Returning')}.")
if len(sig_ch):
    lines.append("- **Channels significantly different from the rest:** " + "; ".join(
        f"{r_.segment} {pct(r_.segment_rate)} ({pp(r_.difference)})" for r_ in sig_ch.itertuples()) + ".")
lines.append("- **Closed funnel, share continuing at each step:** " + "; ".join(
    f"{r_.step} {r_.rate_from_previous:.1%}" for r_ in steps.itertuples()) + ".")
lines.append(f"- **Purchases outside the closed funnel:** {purchases_skipping:.0%} of sessions with a purchase skipped an earlier step in that session.")
lines.append(f"- **Biggest funnel loss:** {int(biggest_leak['lost_from_previous']):,} sessions lost at "
             f"'{biggest_leak['step']}' ({biggest_leak['rate_from_previous']:.1%} continue). Lowest step rate: "
             f"'{lowest_step['step']}' at {lowest_step['rate_from_previous']:.1%}.")
for name_, tt in [("a device differs from desktop", step_tests), ("new visitors differ from returning", user_step_tests)]:
    sig_steps = tt[tt["significant"]]
    lines.append(f"- **Funnel steps where {name_}:** " + ("; ".join(
        f"{r_.segment} {r_.step.lower()} {pct(r_.segment_rate)} vs {pct(r_.comparison_rate)} ({pp(r_.difference)})"
        for r_ in sig_steps.itertuples()) if len(sig_steps) else "none") + ".")
lines.append(f"- **Cart abandonment:** {SITE_ABANDON:.1%} of sessions that added to cart did not buy in the same session.")
for t_ in cart_tests.itertuples():
    lines.append(f"- **Cart completion, {t_.segment} vs {t_.compared_with}:** {pct(t_.segment_rate)} vs {pct(t_.comparison_rate)} "
                 f"({pp(t_.difference)}, adjusted p = {t_.p_adjusted:.3g}{'' if t_.significant else ', not significant'}).")
lines.append("- **Channels per session vs by first touch:** " + "; ".join(
    f"{r_.channel} {pct(r_.session_conversion)} per session, {pct(r_.visitor_to_buyer)} of first-touch visitors bought"
    for r_ in attribution.itertuples()) + ".")
sig_aov = aov_tests[aov_tests["significant"] == True]
lines.append("- **Order value:** " + ("; ".join(
    f"{r_.segment} ${r_.mean_order_value:,.0f} vs {r_.compared_with} ({'+' if r_.difference >= 0 else '-'}${abs(r_.difference):,.0f})"
    for r_ in sig_aov.itertuples()) if len(sig_aov) else "no significant differences by device, visitor type or channel") + ".")
lines.append(f"- **Product data:** {items_per_view_session:.0f} item-list rows per session on product views and {items_per_cart_session:.0f} on adds to cart, so only purchases are used per product.")
lines.append(f"- **Products:** {len(prod)} sold; top {PRODUCTS_FOR_HALF} brought half of item revenue, top 20 brought {TOP20_SHARE:.0%}. Groups: " + "; ".join(
    f"{r_.item_category} {r_.share_of_revenue:.0%}" for r_ in cat.head(6).itertuples()) + ".")
lines.append(f"- **Customers:** {repeat_rate:.1%} of {len(buyers):,} buyers ordered more than once, bringing {repeat_rev_share:.1%} of revenue; "
             f"the top 10% of buyers brought {top10_share:.1%}.")
for o_ in opportunities.head(6).itertuples():
    lines.append(f"- **Opportunity, {o_.opportunity}:** {usd(o_.extra_revenue_25pct)} over the three months at 25% of the gap "
                 f"({usd(o_.extra_revenue_10pct)} to {usd(o_.extra_revenue_50pct)}); counted revenue was {usd(period_revenue)}.")
summary = "\n".join(lines)
(OUT / "summary.md").write_text("# Analysis summary\n\n" + summary + "\n")
display(Markdown(summary))